# Kansas Flood Inundation Mapping

This notebook demonstrates how to use Python to map flood inundation depth using tiled FLDPLN libraries. Inundation maps can be generated using current stage observation from [AHPS](https://water.weather.gov/ahps) and [USGS](https://waterdata.usgs.gov/nwis), historical flood stage from AHPS, and user provided stage. This notebook is based on the script task_lib2map_gauge_fc.py.

To be added:
* forecast stage from AHPS
* postcast stage
* special flood maps

## Download Notebook, Scripts and FLDPLN Libraries

* Download [tools.zip](https://itprdkarsap.home.ku.edu/download/fldpln/tools.zip) and unzip it
* Download Kansas [FLDPLN libraries](https://itprdkarsap.home.ku.edu/download/fldpln/KS_tiled10km_snz_library.zip) (be patient, as it will take a while to dowload the data) and unzip it.
* Download stream [segment shapefiles](https://itprdkarsap.home.ku.edu/download/fldpln/library_segments.zip) and unzip it.

## Create Python Enviroment

Use a Python package management tool (such as conda) and the fldpln.yaml file in the tools.zip to create a Python environment to run notebooks aand scripts. 

## Import Modules

In [2]:
import sys
import time
#import os # Added
#import geopandas as gpd # Added
#import pandas as pd # Added
from dask.distributed import Client, LocalCluster
from dask import visualize

# local tool/script folder
fldplnToolFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\FloodMaps\Map_Aug2016\Tools'

# Add the tool/script folder to sys.path to access the fldpln modules
sys.path.append(fldplnToolFolder) 
# fldpln module
from fldpln import *
from fldpln_gauge import *

## Set FLDPLN Libraries

Both zipped FLDPLN [libraries](https://itprdkarsap.home.ku.edu/download/fldpln/KS_tiled10km_snz_library.zip) and stream [segment shapefiles](https://itprdkarsap.home.ku.edu/download/fldpln/library_segments.zip) can be downloaded from KU server. You can also download the small [spring library](https://itprdkarsap.home.ku.edu/download/fldpln/KS_tiled10km_snz_library_spring.zip) for a test run.

In [3]:
# library segment shapefile
segShp = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\FloodMaps\Map_Aug2016\Library_Segments\library_segments.shp'

# tiled library folder
# libFolder = r'D:\xingong\fldpln_libs'
libFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\FloodMaps\Map_Aug2016\ARB_tiled10km_snz_library'

# select libraries to map
# allLibNames = [ 'bigblue','chik_saltfork_medlodg','cotneo','ksrv','litark',
#                'lowark','lowrep','mdc','midkan','midsol',
#                'morv','nemaha','neosho','ninnescah','osagemarmaton',
#                'smoksalsol','spring','upark','upkan','uprep',
#                'upsal','upsmoky','upsol','verdigriscaney','walnut']
                
libNames = ['Amite'] 
# libNames = ['mdc', 'neosho', 'osagemarmaton', 'spring', 'verdigriscaney', 'walnut']
# libNames = allLibNames

## Get Stages

There are several ways/stages that can be used to map inndation depth:
* Current stage observations that can be accessed from AHPS and USGS Web Services
* Historical flood stages available at some AHPS gauges
* User provided stages

### User Provided Stages

Excel file fim_gauges.xlsx contains stream gauges that can be used for flood mapping in the libraries. Those gauges are obtained from either USGS or AHPS. User can use those gauges for historical flood event mapping. Some gauge flood events can be found from the gauges' historic crest tables provided by [NWS](https://water.weather.gov/ahps2/index.php?wfo=sgf). 

User gauge stage Excel file should have at least the fields: stationid, x, y, stage_elevation. The vertical datum for stage_elevation is assumed as NAVD88. File fim_gauges.xlsx contains gauge vertical datum and conversion information.

#### Amite River 2016 August Flood Event

Excel file AmiteGauges.xlsx provides an example of using user provided gauge stages to map iundation depth. The gauge file (sheet AllGauges) is extracted from the fim_gauges.xlsx file and uncessary fields are deleted. The 33 stages for the flood event on (or around) August 11-13, 2016 are obtained from the gauges' historic crest tables provided by [NWS](https://water.weather.gov/ahps2/index.php?wfo=sgf). 

In [4]:
# User gauge stage file name
gaugeStageFileName = 'AmiteGauges.xlsx'
sheetName = 'Available' # stages at 3 gauges on 12/28/2015
# sheetName = 'BXTK1_Major' # BXTK1 Major flood stage from https://water.weather.gov/ahps2/hydrograph.php?wfo=sgf&gage=bxtk1

# read gauge file
gaugeStages = pd.read_excel(gaugeStageFileName, sheet_name=sheetName) # 3 gauges
print(gaugeStages)

# output map folder name
outMapFolderName = sheetName.lower()

    stationid                                               name organization  \
0     7380126              Henderson Bayou near Port Vincent, LA         USGS   
1     7380102  Welsh Gully at J. Broussard Rd nr Prairieville...         USGS   
2     7378745                Alligator Bayou near Kleinpeter, LA         USGS   
3     7378746  Bayou Manchac at Alligator B. near Kleinpeter, LA         USGS   
4     7380120                    Amite River at Port Vincent, LA         USGS   
5     7377300                        Amite River at Magnolia, LA         USGS   
6     7380200             Amite River near French Settlement, LA         USGS   
7     7380107  Muddy Creek at Manchac Acres Rd near Oak Grove...         USGS   
8     7378050  Comite R. at Greenwell Spg Rd near Baton Rouge...         USGS   
9     7380215            Amite River at Hwy 22 near Maurepas, LA         USGS   
10    7379960  Dawson Cr. at Bluebonnet Blvd near Baton Rouge...         USGS   
11    7377500               

## Prepare Output Map Folder

* outputFolder: a folder where the map folder is saved and temporary files are saved under the 'scratch' folder under the output folder.
* mapFolderName: optional, when provided it supersedes the default folder name based on stageType

In [5]:
# Set flood map output folder
# outputFolder = r'D:\xingong\maps'
outputFolder = r'C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\FloodMaps\Map_Aug2016\OutputMap_Aug11_Aug13_2016'

# Create folders for storing temp and output map files
outMapFolder,scratchFolder = CreateFolders(outputFolder,'scratch',outMapFolderName)

## Set Mapping Parameters

Whether to mosaic tiles as single COG file and whether use a Dask local cluster for mapping.

In [6]:
# whether mosaci tiles as a single COG
mosaicTiles = True #True #False

# Using LocalCluster by default
useLocalCluster = False
numOfWorkers = round(0.8*os.cpu_count())

## Mapping Flood Inundation Depth


In [7]:
# show mapping info
print(f'Tiled FLDPLN library folder: {libFolder}')
print(f'Map folder: {outMapFolder}')
print(f'Libraries mapped: {libNames}')

# check running time
startTimeAllLibs = time.time()

# create a local cluster to speed up the mapping. Must be run inside "if __name__ == '__main__'"!!!
if useLocalCluster:
    # cluster = LocalCluster(n_workers=4,processes=False)
    try:
        print('Start a LocalCluster ...')
        # NOTE: set worker space (i.e., local_dir) to a folder that the LocalCluster can access. When run the script through a scheduled task, 
        # the system uses C:\Windows\system32 by default, which a typical user doesn't have the access!
        cluster = LocalCluster(n_workers=numOfWorkers, memory_limit='32GB', local_dir=r"C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\FloodMaps\Map_Aug2016\Tools") # for KARS production server (192G RAM & 8 cores)
        # cluster = LocalCluster(n_workers=numOfWorkers,processes=False) # for KARS production server (192G RAM & 8 cores)
        # print('Watch workers at: ',cluster.dashboard_link)
        print(f'Number of workers: {numOfWorkers}')
        client = Client(cluster)
        # print scheduler info
        # print(client.scheduler_info())
    except:
        print('Cannot create a LocalCLuster!')
        useLocalCluster = False

# dict to store lib processing time
libTime={}


# print('Snap gauges to FSPs ...')
# snap gauges to FSPs on-the-fly
print(f'Number of gauges: {len(gaugeStages.index)}')
gaugeFspDf = SnapGauges2Fsps(libFolder,libNames,gaugeStages,snapDist=350,gaugeXField='x',gaugeYField='y',fspColumns=['FspX','FspY'])
print(f'Number of gauges snapped to FSPs: {len(gaugeFspDf.index)}')
print(gaugeFspDf)
gaugeFspDf.to_csv(os.path.join(outputFolder,f'{outMapFolderName}_SnappedGauges.csv'), index=False)

# narrow libs need to be mapped
libsHaveGauges = gaugeFspDf['lib_name'].drop_duplicates().tolist()
print(f'Libs have gauges: {libsHaveGauges}')

# keep only necessary columns for gauge FSPs
gaugeFspDf = gaugeFspDf[['stage_elevation','lib_name','FspX','FspY']] # Note that 'lib_name','FspX', 'FspY' together uniquely identify a FSP!!!
print(f'Gauge FSPs: {len(gaugeFspDf)}')

# prepare dfs for saving interpolated FSPs and their segment IDs
fspCols = fspColumnNames + ['Dof']
segIdCols = ['SegId','LibName']
fsps = pd.DataFrame(columns=fspCols)
segIds =pd.DataFrame(columns=segIdCols)

# map each library


for libName in libsHaveGauges:
    # check running time
    startTime = time.time()

    # interpolate DOF for the gauges
    # print('Interpolate FSP DOF using gauge DOF ...')
    fspDof = EstimateFspDofFromGaugeNew(libFolder,libName,gaugeFspDf)
    # print(fspDof)

    # Keep interpolated FSP DOF for saving later
    fspFile = os.path.join(libFolder, libName, fspFileName)
    fspDf = pd.read_csv(fspFile) 
    fspDf = pd.merge(fspDf,fspDof,how='inner',on=['FspId'])
    fsps = fsps.append(fspDf, ignore_index=True)
    
    # Keep FSP segment IDs for saving later
    t =  pd.DataFrame(fspDf['SegId'].drop_duplicates().sort_values())
    t['LibName'] = libName
    segIds = segIds.append(t, ignore_index=True)
    
    # mapping flood depth
    if useLocalCluster:
        print(f'Map {libName} using LocalCLuster ...')
        # generate a DAG
        dag,dagRoot=MapFloodDepthWithTilesAsDag(libFolder,libName,'snappy',outMapFolder,fspDof,aoiExtent=None)
        if dag is None:
            tileTifs = None
        else:
            # visualize DAG
            # visualize(dag)
            # Compute DAG
            tileTifs = client.get(dag, dagRoot)
    else:
        print(f'Map {libName} ...')
        tileTifs = MapFloodDepthWithTiles(libFolder,libName,'snappy',outMapFolder,fspDof,aoiExtent=None)

    # Mosaic all the tiles from a library into one tif file
    if mosaicTiles and not(tileTifs is None):
        print('Mosaic tile maps ...')
        mosaicTifName = libName+'_'+outMapFolderName+'.tif'
        # Simplest implementation, may crash with very large raster
        MosaicGtifs(outMapFolder,tileTifs,mosaicTifName,keepTifs=False)
    
    # check time
    endTime = time.time()
    usedTime = round((endTime-startTime)/60,3)
    libTime[libName] = usedTime
    # print(f'{libName} processing time (minutes):', usedTime)

# save interpolated FSP DOF and their segment IDs
fsps.to_csv(os.path.join(outputFolder,f'{outMapFolderName}_interpolated_FSP_DOF.csv'), index=False)
# segIds.to_csv(os.path.join(outputFolder,f'{outMapFolderName}_interpolated_SegId.csv'), index=False)

# turn interpolated sgements into a shapefile
segs = gpd.read_file(segShp)
# print(segs)
# join by two fields: SegId and LibName
segDf = pd.merge(segs,segIds,how='inner',on=['SegId','LibName'])
# print(segDf)
# write segments as a shapefile
segDf.to_file(os.path.join(outputFolder,f'{outMapFolderName}_interpolated_SegId.shp'))

# Show processing time
# Individual lib processing time
print('Individual library mapping time:', libTime)
# total time
endTimeAllLibs = time.time()
print('Total processing time (minutes):', round((endTimeAllLibs-startTimeAllLibs)/60,3))

#
# Shutdown local clusters
#
if useLocalCluster:
    print('Shutdown LocalCluster ...')
    cluster.close()
    client.shutdown()
    client.close()
    useLocalCluster = False

Tiled FLDPLN library folder: C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\FloodMaps\Map_Aug2016\ARB_tiled10km_snz_library
Map folder: C:\Users\jsong33\Desktop\Research\FIM\FLDPLN_FIM\Amite_h15_dh1_Resample\FloodMaps\Map_Aug2016\OutputMap_Aug11_Aug13_2016\available
Libraries mapped: ['Amite']
Number of gauges: 33
Number of gauges snapped to FSPs: 14
    index  stationid                                               name  \
4       4    7380120                    Amite River at Port Vincent, LA   
5       5    7377300                        Amite River at Magnolia, LA   
8       8    7378050  Comite R. at Greenwell Spg Rd near Baton Rouge...   
10     11    7377500                 Comite River near Olive Branch, LA   
11     12    7377870   Comite River at Hooper Road near Baton Rouge, LA   
12     13    7377000                    Amite River near Darlington, LA   
13     14    7377600  Comite R. at Pt. Hudson-Pride Rd near Milldale...   
14     15    7380101  

c:\ProgramData\anaconda3\envs\FLDPLN\lib\site-packages\rio_cogeo\cogeo.py:157: IncompatibleBlockRasterSize: Block Size are bigger than raster sizes. Setting blocksize to 128
  IncompatibleBlockRasterSize,
c:\ProgramData\anaconda3\envs\FLDPLN\lib\site-packages\rio_cogeo\cogeo.py:157: IncompatibleBlockRasterSize: Block Size are bigger than raster sizes. Setting blocksize to 256
  IncompatibleBlockRasterSize,
c:\ProgramData\anaconda3\envs\FLDPLN\lib\site-packages\rio_cogeo\cogeo.py:157: IncompatibleBlockRasterSize: Block Size are bigger than raster sizes. Setting blocksize to 64
  IncompatibleBlockRasterSize,
c:\ProgramData\anaconda3\envs\FLDPLN\lib\site-packages\rio_cogeo\cogeo.py:146: IncompatibleBlockRasterSize: Raster has dimension < 64px. Output COG cannot be tiled and overviews cannot be added.
  IncompatibleBlockRasterSize,


Actual tiles mapped: ['C:\\Users\\jsong33\\Desktop\\Research\\FIM\\FLDPLN_FIM\\Amite_h15_dh1_Resample\\FloodMaps\\Map_Aug2016\\OutputMap_Aug11_Aug13_2016\\available\\Amite_tile_1.tif', 'C:\\Users\\jsong33\\Desktop\\Research\\FIM\\FLDPLN_FIM\\Amite_h15_dh1_Resample\\FloodMaps\\Map_Aug2016\\OutputMap_Aug11_Aug13_2016\\available\\Amite_tile_2.tif', 'C:\\Users\\jsong33\\Desktop\\Research\\FIM\\FLDPLN_FIM\\Amite_h15_dh1_Resample\\FloodMaps\\Map_Aug2016\\OutputMap_Aug11_Aug13_2016\\available\\Amite_tile_3.tif', 'C:\\Users\\jsong33\\Desktop\\Research\\FIM\\FLDPLN_FIM\\Amite_h15_dh1_Resample\\FloodMaps\\Map_Aug2016\\OutputMap_Aug11_Aug13_2016\\available\\Amite_tile_4.tif', 'C:\\Users\\jsong33\\Desktop\\Research\\FIM\\FLDPLN_FIM\\Amite_h15_dh1_Resample\\FloodMaps\\Map_Aug2016\\OutputMap_Aug11_Aug13_2016\\available\\Amite_tile_5.tif', 'C:\\Users\\jsong33\\Desktop\\Research\\FIM\\FLDPLN_FIM\\Amite_h15_dh1_Resample\\FloodMaps\\Map_Aug2016\\OutputMap_Aug11_Aug13_2016\\available\\Amite_tile_6.tif', 